# Starting off
Minimal examples to test my understanding of the LLM and the tools.

In [1]:
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import torch
import pandas as pd

df = pd.read_csv('./knowledge/clean_youtube_watch_log.csv')
title = df['video_title'].iloc[0]
thumbnail = Image.open(f'./knowledge/thumbnails/{df["video_id"].iloc[0]}.jpg')

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
def embed_text(text: str):
    '''Embed a text using CLIP'''
    inputs = proc(text=[text], return_tensors="pt", truncation=True, padding=True)  # WARNING! Truncation may lead to loss of information for long texts
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True) # Normalize 

def embed_image(image: Image.Image):
    '''Embed an image using CLIP'''
    inputs = proc(images=image, return_tensors="pt")
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
    return emb / emb.norm(dim=-1, keepdim=True) # Normalize

def similarity(a: torch.Tensor, b: torch.Tensor):
    '''Calculate cosine similarity between two embeddings'''
    return (a @ b.T).item()

In [3]:
# text / text
sim = similarity(embed_text("cat"), embed_text("dog"))
print(f"Similarity between 'cat' and 'dog': {sim:.4f}")

# text / image
img = Image.open("./knowledge/thumbnails/-n8GYm35FeA.jpg")
sim = similarity(embed_text("car"), embed_image(img))
print(f"Similarity between 'car' and thumbnail: {sim:.4f}")

Similarity between 'cat' and 'dog': 0.9122
Similarity between 'car' and thumbnail: 0.2566


# Indexing the embeddings of whole dataset in FAISS

In [ ]:
import faiss
from tqdm.notebook import tqdm

def build_index(texts: list[str], images: list[Image.Image]):
    text_vecs = []
    img_vecs = []

    # Embed all texts and images
    for t, img in tqdm(zip(texts, images), total=len(texts), desc="Embedding Image / Text pairs"):
        text_vecs.append(embed_text(t))
        img_vecs.append(embed_image(img))

    # Convert to numpy arrays
    text_vecs = torch.cat(text_vecs).cpu().numpy()
    img_vecs  = torch.cat(img_vecs).cpu().numpy()

    # Find embedding dimension
    d = text_vecs.shape[1]

    # Createa and populate the indices
    index_text = faiss.IndexFlatIP(d)  # cosine (because normalized)
    index_img  = faiss.IndexFlatIP(d)
    index_text.add(text_vecs)
    index_img.add(img_vecs)

    return index_text, index_img

In [5]:
texts = df['video_title'].tolist()
images = [Image.open(f'./knowledge/thumbnails/{vid_id}.jpg') for vid_id in df['video_id']]
index_text, index_img = build_index(texts, images)

Embedding:   0%|          | 0/987 [00:00<?, ?it/s]

## Implement similarity search for text and images

In [6]:
def search_index(query: str, index: faiss.Index, k=5):
    q_vec = embed_text(query).cpu().numpy()
    scores, ids = index.search(q_vec, k)
    return scores[0], ids[0]

In [7]:
query = "funny cat"

# Search for similar video titles
scores, ids = search_index(query, index_text)
print(f"Top results for query '{query}':")
for score, idx in zip(scores, ids):
    print(f"  - {df['video_title'].iloc[idx]:60} (score: {score:6.4f})")

# Search for similar thumbnails
scores, ids = search_index(query, index_img)
print(f"\nTop thumbnails for query '{query}':")
for score, idx in zip(scores, ids):
    print(f"  - {df['video_title'].iloc[idx]:60} (score: {score:6.4f})")

Top results for query 'funny cat':
  - NOT ME....                                                   (score: 0.8451)
  - Little kitten thinks I'm his mom                             (score: 0.8306)
  - I tech solution                                              (score: 0.8296)
  - Cats vs Zombies!                                             (score: 0.8201)
  - Travel Inspiration                                           (score: 0.8191)

Top thumbnails for query 'funny cat':
  - 子猫　ミルク飲むのはやっ！                                                (score: 0.2855)
  - Peekaboo Playground | Kids Songs | Super Simple Songs        (score: 0.2814)
  - NATURE CAT | Super Secret Spy Mission! | PBS KIDS            (score: 0.2813)
  - 3 Week Old Kitten Feeding Time                               (score: 0.2766)
  - CBeebies Songs | Andy's Cool Cats Rap                        (score: 0.2737)


# Indexing the embeddings of whole dataset in ChromaDB

In [3]:
import chromadb
from tqdm.notebook import tqdm

def build_index(ids, titles, thumbnails):
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.create_collection(name="youtube_videos")
    
    # Embed all texts and images
    vecs = []
    for t, img in tqdm(zip(titles, thumbnails), total=len(titles), desc="Embedding Image / Text pairs"):
        text_emb = embed_text(t)
        img_emb  = embed_image(img)
        # fusion: 70% text, 30% image
        vecs.append(0.7 * text_emb + 0.3 * img_emb)

    # Convert to numpy array
    vecs = torch.cat(vecs).cpu().numpy()

    # Add embeddings to the collection
    collection.add(
        ids=ids,            # identifier for each entry (e.g., video ID)
        embeddings=vecs,    # list of embeddings (fused text + image)
        documents=titles    # optional: store the original titles as metadata
    )

    return collection

def load_index():
    client = chromadb.PersistentClient(path="./chroma_db")
    collection = client.get_collection(name="youtube_videos")
    return collection

E0000 00:00:1777113182.287782  275711 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1777113182.288893  275711 instrument.cc:563] Metric with name 'grpc.resource_quota.calls_rejected' registered more than once. Ignoring later registration.
E0000 00:00:1777113182.288903  275711 instrument.cc:563] Metric with name 'grpc.resource_quota.connections_dropped' registered more than once. Ignoring later registration.
E0000 00:00:1777113182.288908  275711 instrument.cc:563] Metric with name 'grpc.resource_quota.instantaneous_memory_pressure' registered more than once. Ignoring later registration.
E0000 00:00:1777113182.288910  275711 instrument.cc:563] Metric with name 'grpc.resource_quota.memory_pressure_control_value' registered more than once. Ignoring later registration.


In [6]:
ids = list(dict.fromkeys(df['video_id']))       # ignore duplicates
titles = [df[df['video_id'] == vid]['video_title'].iloc[0] for vid in ids]
images = [Image.open(f'./knowledge/thumbnails/{vid}.jpg') for vid in ids]

# index = build_index(ids, titles, images)
index = load_index()

## Implement similarity search for text and images

In [11]:
def search_index(query: str, index: chromadb.Collection, k=5): 
    q = embed_text(query).squeeze().tolist()

    res = index.query(
        query_embeddings=[q],
        n_results=k
    )

    return res["ids"][0], res["documents"][0], res["distances"][0]

In [21]:
query = "funny cat"

# Search for similar titles
ids, titles, distances = search_index(query, index)
print(f"Top results for query '{query}':")
for id, title, distance in zip(ids, titles, distances):
    print(f"  - {title:60} (distance: {distance:6.4f}, id: {id})")

Top results for query 'funny cat':
  - NOT ME....                                                   (distance: 0.3622, id: 56aL3NgKT5s)
  - Little kitten thinks I'm his mom                             (distance: 0.3692, id: hew_NNBZTXM)
  - I tech solution                                              (distance: 0.3933, id: WloR27rFcAA)
  - VMH                                                          (distance: 0.3985, id: Ywmge3X5zMI)
  - Cats vs Zombies!                                             (distance: 0.4012, id: bcENL_Vdoqw)
